# Seeing Conditional Probability in Action

## Learning Objectives

By completing this notebook, you will:

1. Observe how a single added sentence dramatically shifts model output across multiple domains
2. Connect the observable behaviour to the conditional probability mechanism from Guide 01
3. Identify the pattern of prior dominance in real model outputs
4. Write your own condition-specified prompts and observe their effect

## Prerequisites

- Guide 01: Autoregressive Generation
- Guide 02: Prior Dominance
- `ANTHROPIC_API_KEY` set in your environment

## Estimated Time: 15 minutes

---

## Setup

We use the Anthropic Python SDK. The `client.messages.create()` call sends a prompt to Claude and returns the completion. We will use a low temperature (0.3) to make the prior dominance effect visible and consistent — if the model defaults to training priors, it will do so consistently at low temperature.

In [ ]:
import anthropic
import os

# Verify the API key is available before running any cells
assert os.getenv("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY environment variable not set. "
    "Export it before running this notebook: export ANTHROPIC_API_KEY=sk-ant-..."
)

client = anthropic.Anthropic()

print("Anthropic client initialised.")

In [ ]:
def ask(prompt: str, temperature: float = 0.3, max_tokens: int = 300) -> str:
    """
    Send a prompt to Claude and return the text response.

    We use a fixed model and low temperature so that differences between
    responses are driven by the prompt, not by randomness.

    Parameters
    ----------
    prompt : str
        The full prompt text to send.
    temperature : float
        Sampling temperature. Low values (0.0–0.3) make the output more
        deterministic, which makes the prior dominance effect clearer.
    max_tokens : int
        Maximum tokens to generate. We keep this short so responses are
        comparable side by side.

    Returns
    -------
    str
        The model's text response.
    """
    message = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text


def compare(label_a: str, prompt_a: str, label_b: str, prompt_b: str) -> None:
    """
    Run two prompts and print their outputs side by side.

    This makes it easy to see how adding or removing a condition shifts
    the model's output. We call the API sequentially so rate limits are
    not exceeded.
    """
    print(f"{'─' * 70}")
    print(f"PROMPT A — {label_a}")
    print(f"{'─' * 70}")
    print(prompt_a)
    print()
    response_a = ask(prompt_a)
    print("RESPONSE A:")
    print(response_a)

    print()
    print(f"{'─' * 70}")
    print(f"PROMPT B — {label_b}")
    print(f"{'─' * 70}")
    print(prompt_b)
    print()
    response_b = ask(prompt_b)
    print("RESPONSE B:")
    print(response_b)
    print(f"{'═' * 70}")
    print()


print("Helper functions defined.")

---

## Example 1: Tax Advice

**What we are observing:** Jurisdiction and entity type are among the most powerful conditions in any legal or financial prompt. The model's prior defaults to the most common jurisdiction (US) and the most common filing situation (personal return). We will observe how adding a single sentence that specifies UK + corporation tax completely changes the response.

**Prediction before running:** Prompt A will produce advice about IRS penalties and US tax law. Prompt B will produce advice about HMRC and UK corporation tax. Both will be delivered with equal confidence.

In [ ]:
compare(
    label_a="Prior-dominated (no jurisdiction)",
    prompt_a="""My client filed their tax return late. 
What penalties should they expect?""",
    label_b="Condition-specified (UK corporation tax, 2-year delay)",
    prompt_b="""My client is a UK limited company. Their 2024 corporation tax return 
has not been filed and is now being submitted in 2026, two years after the deadline. 
The company was not dissolved but was dormant during the intervening period. 
What HMRC penalties should they expect?""",
)

**Observe:**

- Does Prompt A mention the IRS or US tax law? Does Prompt B shift to HMRC?
- Does the two-year delay change the penalty structure?
- Does either response hedge about jurisdiction?

---

## Example 2: Medical Advice

**What we are observing:** Medical prompts have strong priors around age group, comorbidities, and country of practice. We will see how the model's default recommendation for a drug dosage changes when we specify a paediatric patient with a contraindication.

In [ ]:
compare(
    label_a="Prior-dominated (unspecified patient)",
    prompt_a="""A patient needs pain relief. What is the recommended ibuprofen dose 
and any important precautions?""",
    label_b="Condition-specified (paediatric, renal impairment)",
    prompt_b="""The patient is a 7-year-old child weighing 22 kg with mild renal impairment 
(eGFR 45 mL/min/1.73m²). They need pain relief for post-surgical discomfort. 
What is the recommended ibuprofen dose and any important precautions?""",
)

**Observe:**

- Does Prompt A give an adult dose? Does Prompt B adjust for weight and age?
- Does renal impairment change the recommendation in Prompt B?
- Does the model flag caution about ibuprofen use in renal impairment when the condition is specified?

---

## Example 3: Code Review

**What we are observing:** Code prompts default to greenfield projects with the most recent library versions. A legacy codebase with version constraints requires fundamentally different advice. We will observe how specifying the Python and Django version changes the recommendations.

In [ ]:
compare(
    label_a="Prior-dominated (no version context)",
    prompt_a="""How should I add async support to my Django views for a 
database-heavy endpoint?""",
    label_b="Condition-specified (pinned versions, production constraint)",
    prompt_b="""We are running Django 3.2 LTS on Python 3.8 in production. 
Upgrading is not currently possible — the next upgrade window is 6 months away. 
We need to add async support to a database-heavy endpoint that currently blocks 
for 2–3 seconds. How should we approach this given our version constraints?""",
)

**Observe:**

- Does Prompt A recommend Django 4.x async views or `asyncpg`?
- Does Prompt B acknowledge Django 3.2's limited async ORM support?
- Does Prompt B suggest workarounds compatible with Python 3.8 and Django 3.2?

---

## Example 4: Business Strategy

**What we are observing:** Business strategy prompts default to a well-resourced company in a mature market with standard growth options. Constraints — size, market structure, regulatory environment — fundamentally change what strategies are available. We will see how the model's pricing strategy advice changes when the company is a small player in a regulated market.

In [ ]:
compare(
    label_a="Prior-dominated (generic company)",
    prompt_a="""What pricing strategy should we use to grow market share 
in a competitive market?""",
    label_b="Condition-specified (regulated utility, small player)",
    prompt_b="""We are a small independent water utility in England, regulated by Ofwat 
under a price cap review cycle. We serve 45,000 households. We cannot freely set prices — 
they are approved by the regulator every 5 years. Our next Price Review is PR29. 
How should we approach pricing strategy to grow our allowed revenue while meeting 
Ofwat's efficiency and service quality requirements?""",
)

**Observe:**

- Does Prompt A recommend tactics like freemium, penetration pricing, or dynamic pricing?
- Does Prompt B engage with Ofwat's regulatory framework and price review cycles?
- Does Prompt B acknowledge that standard competitive pricing strategies do not apply?

---

## Reflection: What Shifted the Distribution?

Across all four examples, the same mechanism produced the different outputs: adding conditions shifted the conditional probability distribution $P(\text{output} \mid \text{prompt})$ from "most typical training-data scenario" to "your actual scenario."

Let us verify one more important property: the shift is caused by the added conditions specifically, not by the response simply being longer or more detailed.

In [ ]:
# Control: a longer prompt that does not add the critical conditions
compare(
    label_a="Longer but still prior-dominated (no critical conditions)",
    prompt_a="""I have a client who is an experienced professional and came to me 
with a tax question that I think is quite important and may have significant 
financial implications for them. They filed their tax return late and are concerned 
about what penalties they might face. This has been weighing on them for a while. 
What penalties should they expect?""",
    label_b="Shorter but condition-specified (critical conditions only)",
    prompt_b="""UK limited company. Corporation tax return filed 2 years late. 
Company was dormant during the delay. What HMRC penalties apply?""",
)

**Observe:**

- Does the longer Prompt A still give US-centric advice?
- Does the shorter Prompt B still produce UK-specific advice?

This confirms that **length does not determine accuracy — conditions do**. Adding words that do not specify the conditions that distinguish your situation from the typical case does not help. Adding the right conditions in even a few words shifts the distribution substantially.

---

## Exercise: Your Domain

Now apply this to a domain you actually work in.

**Task:** Write two prompts — one underspecified (prior-dominated) and one condition-specified — for a question relevant to your work. Run them with `compare()` and observe the difference.

Before running, write down:
1. What world does your underspecified prompt assume by default?
2. What conditions did you add, and what world do they specify?
3. What difference do you predict between the two outputs?

In [ ]:
# Your prediction (fill this in before running):
# Default world assumed: ___
# Conditions added: ___
# Predicted difference: ___

your_prompt_a = """<write your underspecified prompt here>"""

your_prompt_b = """<write your condition-specified prompt here>"""

compare(
    label_a="My prior-dominated prompt",
    prompt_a=your_prompt_a,
    label_b="My condition-specified prompt",
    prompt_b=your_prompt_b,
)

---

## Summary

### Key Takeaways

1. **Prior dominance is visible in real outputs.** The same question with different conditions produces substantially different answers, delivered with equal confidence.

2. **Conditions shift distributions; length does not.** A short, condition-rich prompt outperforms a long, condition-poor prompt every time.

3. **The effect is domain-independent.** Tax, medicine, code, and business strategy all show the same mechanism: underspecified prompts produce generic answers, condition-specified prompts produce situation-appropriate answers.

4. **This is the core skill.** Every technique in this course is a structured method for identifying and specifying the conditions that shift the model's distribution toward the correct answer for your specific situation.

### What's Next

**Exercise 01:** Given five real prompts that produce generic answers, diagnose which conditions are missing and write improved versions. This is in `exercises/01_identify_missing_conditions.md`.

**Module 1:** Prompts as Bayesian Evidence — a systematic framework for mapping any task to the conditions that most shift the probability distribution.

### Troubleshooting

- **`AuthenticationError`:** Your `ANTHROPIC_API_KEY` is not set or is invalid. Check with `echo $ANTHROPIC_API_KEY`.
- **`RateLimitError`:** You have exceeded your API quota. Add `time.sleep(1)` between calls or reduce `max_tokens`.
- **Responses look similar:** Try increasing the contrast between your two prompts, or lower the temperature to 0.1.